# tcgews — Operational Genesis Forecast Demo

**Predictive Tropical Cyclogenesis Using Machine Learning: A Novel Approach to Early Warning Systems**
(Nikolay Todorov, Chapman University) · [Repository](https://github.com/niko-todorov/tcgews) · DOI [10.5281/zenodo.23204865](https://doi.org/10.5281/zenodo.23204865)

This notebook reproduces one of the operational case studies from Chapter 7 of the dissertation:
it fetches an ERA5 analysis, runs the trained `TemporalTCG` early-warning system, and renders the
genesis-location probability field across the 0–48 h lead window — the same output shown in
Figures 7.1–7.8.

**Runtime:** no GPU required (inference is a single forward pass). *Runtime → Change runtime type → CPU* is fine.

You only need to wire up two things (marked below): the **trained weights** (pulled from Zenodo) and
**ERA5 access** (a free Copernicus CDS key, or a pre-staged sample).

## 1. Set up the environment
Clone the repository and install dependencies from `requirements.txt`.
(Colab already ships `torch`, `numpy`, `pandas`, `scipy`, `scikit-learn`, `matplotlib`, and `xarray`,
so those are skipped; the install mostly fetches `netCDF4`, `cfgrib`/`eccodes`, `cdsapi`,
`ecmwf-opendata`, `tcpyPI`, and `cartopy`.)

In [ ]:
!git clone --depth 1 https://github.com/niko-todorov/tcgews.git
%cd tcgews

In [ ]:
!pip install -q -r requirements.txt
print("\nEnvironment ready.")

## 2. Download the trained model weights
The checkpoints are not in the git history (they are large); they are attached to the Zenodo
record. This cell lists the record's files and downloads any PyTorch checkpoint automatically.

> If nothing downloads, attach the trained `.pt` weights to the Zenodo deposit
> (record `23204865` → *Edit* → upload), then re-run.

In [ ]:
import requests, os
REC = "10.5281/zenodo.23204865".split("zenodo.")[-1]     # -> 23204865
meta = requests.get(f"https://zenodo.org/api/records/{REC}").json()
ckpts = [f for f in meta.get("files", []) if f["key"].lower().endswith((".pt", ".pth", ".ckpt"))]
if not ckpts:
    print("No .pt/.pth/.ckpt found on the Zenodo record. Attach the trained weights to the deposit, "
          "or set a direct URL below and download manually.")
for f in ckpts:
    name, url = f["key"], f["links"]["self"]
    print("Downloading", name, "...")
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(name, "wb") as out:
            for chunk in r.iter_content(1 << 20):
                out.write(chunk)
    print("  saved", name, f"({os.path.getsize(name)/1e6:.1f} MB)")

# NOTE: place the checkpoint where tcg_ews.py expects it (see the script's --help / weights path).
# If tcg_ews.py looks for a specific filename or directory, move/rename here, e.g.:
# os.makedirs("weights", exist_ok=True); os.replace("convlstm_nacsgm_best.pt", "weights/convlstm_nacsgm_best.pt")

## 3. ERA5 access
The reproducible case studies read ERA5 from the Copernicus Climate Data Store (CDS), which needs a
free account + API key (<https://cds.climate.copernicus.eu/>). Paste your key below; it is written to
`~/.cdsapirc` and never stored in the notebook.

> **Prefer a true one-click demo?** Pre-stage the ERA5 inputs for one case as a small file on the
> Zenodo record (or a GitHub release asset) and download it here instead — then no CDS key is needed.

In [ ]:
import getpass, os
CDS_URL = "https://cds.climate.copernicus.eu/api"          # current CDS endpoint
key = getpass.getpass("Paste your CDS API key: ").strip()
with open(os.path.expanduser("~/.cdsapirc"), "w") as f:
    f.write(f"url: {CDS_URL}\nkey: {key}\n")
print("~/.cdsapirc written.")

## 4. Run a case study — Hurricane Erika (2009), Figure 7.3
NACSGM basin; ERA5 analysis 2009-09-02 18:00 UTC; observed genesis 16.3°N, 299.0°E.
The flags (`--taper --ocean-fill --halo 6 --argmax component`) match the dissertation figures.
First run downloads the ERA5 fields (a few minutes).

In [ ]:
!python tcg_ews.py --basin NACSGM --source era5 --time 2009-09-02T18:00 \
    --truth 16.3,299.0 --taper --ocean-fill --halo 6 --argmax component

In [ ]:
# Display the rendered genesis-probability field
from IPython.display import Image, display
import glob
imgs = sorted(glob.glob("**/*20090902*component*.png", recursive=True))
if imgs:
    display(Image(filename=imgs[-1]))
else:
    print("Output image not found — check the run log above (and the model-weights path in step 2).")

## 5. Try other cases
Swap `--time` and `--truth` (and `--basin`) for any case from Table C.2 of the dissertation:

| Fig | Storm (year) | Basin | `--time` | `--truth` (lat, lon °E) |
|-----|--------------|-------|----------|--------------------------|
| 7.1 | Klaus (1984)   | NACSGM | 1984-11-05T18:00 | 14.7, 291.2 |
| 7.2 | Fabian (1991)  | NACSGM | 1991-10-15T00:00 | 18.9, 274.3 |
| 7.3 | Erika (2009)   | NACSGM | 2009-09-02T18:00 | 16.3, 299.0 |
| 7.4 | Allison (1989) | NACSGM | 1989-06-24T18:00 | 27.0, 264.0 |
| 7.5 | Keith (1997)   | WPMM   | 1997-10-27T06:00 | 6.7, 168.7  |
| 7.6 | Rananim (2004) | WPMM   | 2004-08-06T00:00 | 15.3, 136.8 |
| 7.7 | Dujuan (2009)  | WPMM   | 2009-09-02T18:00 | 17.6, 130.1 |
| 7.8 | Kujira (2015)  | WPMM   | 2015-06-19T18:00 | 15.0, 112.2 |

> Historical cases must use `--source era5` — the real-time `--source ifs-opendata` only covers the
> most recent ~2–3 days (ECMWF's open-data rolling window).